<a href="https://colab.research.google.com/github/carlosg2003/IA-7mo-Semestre/blob/master/multiagentemundial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema Multiagente Predictivo y RAG para los Mundiales de Fútbol

Este notebook implementa un flujo multiagente completo para procesar, normalizar, entrenar modelos de Machine Learning y generar un reporte ejecutivo semántico a partir del historial de partidos de la Copa del Mundo (`wc_all_matches.csv`).

### Arquitectura del Sistema
1. **Base de Datos Vectorial (RAG)**: Indexa los partidos utilizando `SentenceTransformers` para búsquedas semánticas.
2. **Agente 1 (Normalizador)**: Limpia y normaliza el dataset, y detecta automáticamente si el problema es de clasificación o regresión.
3. **Agente 2 (Entrenador)**: Entrena 5 algoritmos mediante validación cruzada 5-fold y selecciona el mejor.
4. **Agente 3 (Comunicador)**: Construye el reporte final en español incorporando el contexto recuperado de la base vectorial RAG.

## Paso 1: Instalar dependencias necesarias
Instalamos `sentence-transformers` para la base de datos vectorial local. Google Colab ya tiene instalado `scikit-learn` y `transformers` por defecto.

In [1]:
!pip install sentence-transformers

## Paso 2: Subir el dataset `wc_all_matches.csv`
Ejecuta la siguiente celda para subir el archivo de partidos.

In [2]:
from google.colab import files
import os

uploaded = files.upload()
for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))

if not os.path.exists('wc_all_matches.csv'):
    # Si se subió con otro nombre, renombrarlo
    for filename in uploaded.keys():
        if 'matches' in filename.lower() or 'wc' in filename.lower():
            os.rename(filename, 'wc_all_matches.csv')
            print(f'Renombrado {filename} a wc_all_matches.csv')
            break

Saving wc_all_matches.csv to wc_all_matches (1).csv
User uploaded file "wc_all_matches (1).csv" with length 17249 bytes


## Paso 3: Implementar la Base de Datos Vectorial (VectorDB)
Definimos la clase `VectorDB` que utiliza `SentenceTransformer` para vectorizar descripciones en lenguaje natural de los partidos e indexarlas, permitiendo consultas semánticas locales con similitud de coseno.

In [3]:
import os
import json
import re
import numpy as np

class SimpleTFIDF:
    def __init__(self):
        self.vocab = {}
        self.idf = {}
        self.doc_count = 0

    def _tokenize(self, text):
        return re.findall(r'\w+', text.lower())

    def fit(self, documents):
        self.doc_count = len(documents)
        dfs = {}
        for doc in documents:
            tokens = set(self._tokenize(doc))
            for t in tokens:
                dfs[t] = dfs.get(t, 0) + 1

        self.vocab = {t: idx for idx, t in enumerate(sorted(dfs.keys()))}
        for t, df in dfs.items():
            self.idf[t] = np.log((1 + self.doc_count) / (1 + df)) + 1
        return self

    def transform(self, documents):
        N = len(documents)
        V = len(self.vocab)
        if V == 0:
            return np.zeros((N, 1))

        matrix = np.zeros((N, V))
        for doc_idx, doc in enumerate(documents):
            tokens = self._tokenize(doc)
            if len(tokens) == 0:
                continue
            counts = {}
            for t in tokens:
                if t in self.vocab:
                    counts[t] = counts.get(t, 0) + 1
            for t, count in counts.items():
                tf = count / len(tokens)
                v_idx = self.vocab[t]
                matrix[doc_idx, v_idx] = tf * self.idf[t]
        return matrix

    def fit_transform(self, documents):
        return self.fit(documents).transform(documents)


class VectorDB:
    def __init__(self, model_name='all-MiniLM-L6-v2', db_path='vector_db.json'):
        self.model_name = model_name
        self.db_path = db_path
        self.documents = []
        self.metadatas = []
        self.embeddings = None
        self.use_fallback = False
        self.tfidf = None
        self.model = None

    def _lazy_init(self):
        if self.use_fallback:
            return

        if self.model is None:
            try:
                from sentence_transformers import SentenceTransformer
                print(f"[VectorDB] Cargando SentenceTransformer: {self.model_name}...")
                self.model = SentenceTransformer(self.model_name)
            except Exception as e:
                print(f"[VectorDB] No se pudo cargar SentenceTransformers ({e}). Usando fallback de TF-IDF local...")
                self.use_fallback = True
                self.tfidf = SimpleTFIDF()
                if len(self.documents) > 0:
                    self.embeddings = self.tfidf.fit_transform(self.documents)

    def add_texts(self, texts, metadatas=None):
        self.documents.extend(texts)
        if metadatas:
            self.metadatas.extend(metadatas)
        else:
            self.metadatas.extend([{} for _ in texts])

        self._lazy_init()

        if self.use_fallback:
            self.embeddings = self.tfidf.fit_transform(self.documents)
        else:
            new_embeddings = self.model.encode(texts, show_progress_bar=False)
            if self.embeddings is None:
                self.embeddings = np.array(new_embeddings)
            else:
                self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self, query, k=3):
        if self.embeddings is None or len(self.documents) == 0:
            return []

        self._lazy_init()

        if self.use_fallback:
            query_vector = self.tfidf.transform([query])[0]
            q_norm = np.linalg.norm(query_vector)
            if q_norm == 0:
                q_norm = 1e-10

            dot_products = np.dot(self.embeddings, query_vector)
            norms = np.linalg.norm(self.embeddings, axis=1) * q_norm
            norms = np.where(norms == 0, 1e-10, norms)
            similarities = dot_products / norms
        else:
            query_embedding = self.model.encode([query], show_progress_bar=False)[0]
            dot_products = np.dot(self.embeddings, query_embedding)
            norms = np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(query_embedding)
            norms = np.where(norms == 0, 1e-10, norms)
            similarities = dot_products / norms

        top_k_indices = np.argsort(similarities)[::-1][:k]

        results = []
        for idx in top_k_indices:
            results.append({
                'document': self.documents[idx],
                'metadata': self.metadatas[idx],
                'score': float(similarities[idx])
            })
        return results

    def save(self):
        db_data = {
            'model_name': self.model_name,
            'documents': self.documents,
            'metadatas': self.metadatas,
            'use_fallback': self.use_fallback
        }
        with open(self.db_path, 'w', encoding='utf-8') as f:
            json.dump(db_data, f, ensure_ascii=False, indent=2)

        if self.embeddings is not None:
            np.save(self.db_path + '.npy', self.embeddings)

    def load(self):
        if not os.path.exists(self.db_path):
            return False

        with open(self.db_path, 'r', encoding='utf-8') as f:
            db_data = json.load(f)
            self.model_name = db_data.get('model_name', self.model_name)
            self.documents = db_data.get('documents', [])
            self.metadatas = db_data.get('metadatas', [])
            self.use_fallback = db_data.get('use_fallback', False)

        npy_path = self.db_path + '.npy'
        if os.path.exists(npy_path):
            self.embeddings = np.load(npy_path)

        if self.use_fallback:
            self.tfidf = SimpleTFIDF()
            if len(self.documents) > 0:
                self.tfidf.fit(self.documents)
        return True

## Paso 4: Implementar los Agentes (Normalizador, Entrenador y Comunicador)
Definimos las clases para cada uno de los tres agentes principales.

In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.svm import SVC, SVR

class NormalizerAgent:
    def __init__(self):
        self.numeric_imputer = SimpleImputer(strategy='mean')
        self.categorical_imputer = SimpleImputer(strategy='most_frequent')
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.target_encoder = None
        self.is_classification = None
        self.numeric_cols = []
        self.categorical_cols = []

    def detect_problem_type(self, y):
        if y.dtype == object or y.dtype == bool or str(y.dtype) == 'category':
            return True
        unique_vals = len(np.unique(y.dropna()))
        if unique_vals < 15:
            return True
        return False

    def clean_and_prepare(self, df, target_col):
        print("[Agente 1] Iniciando normalización y limpieza...")
        df_clean = df.drop_duplicates().copy()
        print(f"[Agente 1] Duplicados eliminados. Filas originales: {len(df)}, Filas limpias: {len(df_clean)}")

        if target_col not in df_clean.columns:
            raise ValueError(f"La columna objetivo '{target_col}' no existe en el dataset.")

        self.is_classification = self.detect_problem_type(df_clean[target_col])
        problem_str = "Clasificación" if self.is_classification else "Regresión"
        print(f"[Agente 1] Tipo de problema detectado automáticamente: {problem_str}")

        X = df_clean.drop(columns=[target_col])
        y = df_clean[target_col]

        self.numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

        if len(self.numeric_cols) > 0:
            print(f"[Agente 1] Columnas numéricas a procesar: {self.numeric_cols}")
            X_num_imputed = pd.DataFrame(
                self.numeric_imputer.fit_transform(X[self.numeric_cols]),
                columns=self.numeric_cols,
                index=X.index
            )
            X_num_scaled = pd.DataFrame(
                self.scaler.fit_transform(X_num_imputed),
                columns=self.numeric_cols,
                index=X.index
            )
        else:
            X_num_scaled = pd.DataFrame(index=X.index)

        X_cat_encoded = pd.DataFrame(index=X.index)
        if len(self.categorical_cols) > 0:
            print(f"[Agente 1] Columnas categóricas a procesar: {self.categorical_cols}")
            X_cat_imputed = pd.DataFrame(
                self.categorical_imputer.fit_transform(X[self.categorical_cols].astype(str)),
                columns=self.categorical_cols,
                index=X.index
            )
            for col in self.categorical_cols:
                le = LabelEncoder()
                X_cat_encoded[col] = le.fit_transform(X_cat_imputed[col])
                self.label_encoders[col] = le

        X_clean = pd.concat([X_num_scaled, X_cat_encoded], axis=1)

        y_clean = y.copy()
        if y_clean.isnull().any():
            target_imputer = SimpleImputer(strategy='most_frequent' if self.is_classification else 'mean')
            y_clean = pd.Series(
                target_imputer.fit_transform(y_clean.values.reshape(-1, 1)).flatten(),
                index=y.index,
                name=target_col
            )

        if self.is_classification:
            self.target_encoder = LabelEncoder()
            y_clean = pd.Series(
                self.target_encoder.fit_transform(y_clean.astype(str)),
                index=y.index,
                name=target_col
            )
        else:
            y_clean = pd.Series(y_clean.astype(float), index=y.index, name=target_col)

        print("[Agente 1] Normalización completada con éxito.")
        return X_clean, y_clean, self.is_classification


class TrainerAgent:
    def __init__(self):
        self.best_model = None
        self.best_model_name = None
        self.best_score = -float('inf')
        self.metrics = {}
        self.models_comparison = {}

    def train_and_evaluate(self, X, y, is_classification):
        print("[Agente 2] Iniciando entrenamiento y validación cruzada...")

        if is_classification:
            class_counts = y.value_counts()
            stratify = y if (class_counts > 1).all() else None
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=stratify
            )
        else:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42
            )

        if is_classification:
            candidates = {
                'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
                'Random Forest': RandomForestClassifier(random_state=42),
                'Decision Tree': DecisionTreeClassifier(random_state=42),
                'SVM (SVC)': SVC(probability=True, random_state=42),
                'Gradient Boosting': GradientBoostingClassifier(random_state=42)
            }
            scoring = 'f1_weighted'
            print("[Agente 2] Modelos de clasificación seleccionados (métrica: F1-weighted).")
        else:
            candidates = {
                'Linear Regression': LinearRegression(),
                'Random Forest': RandomForestRegressor(random_state=42),
                'Decision Tree': DecisionTreeRegressor(random_state=42),
                'SVM (SVR)': SVR(),
                'Gradient Boosting': GradientBoostingRegressor(random_state=42)
            }
            scoring = 'r2'
            print("[Agente 2] Modelos de regresión seleccionados (métrica: R²).")

        cv_scores = {}
        for name, model in candidates.items():
            print(f"[Agente 2] Evaluando {name} con 5-Fold CV...")
            try:
                scores = cross_val_score(model, X_train, y_train, cv=5, scoring=scoring)
                mean_score = float(scores.mean())
                cv_scores[name] = mean_score
                self.models_comparison[name] = {
                    'cv_mean_score': mean_score,
                    'cv_scores': [float(s) for s in scores]
                }
                print(f"[Agente 2] {name} - Puntuación Media CV: {mean_score:.4f}")
            except Exception as e:
                print(f"[Agente 2] CV para {name} falló: {e}.")

        self.best_model_name = max(cv_scores, key=cv_scores.get)
        self.best_score = cv_scores[self.best_model_name]
        print(f"[Agente 2] El mejor modelo es: {self.best_model_name} con score {self.best_score:.4f}")

        self.best_model = candidates[self.best_model_name]
        self.best_model.fit(X_train, y_train)

        y_pred = self.best_model.predict(X_test)

        if is_classification:
            acc = float(accuracy_score(y_test, y_pred))
            f1 = float(f1_score(y_test, y_pred, average='weighted', zero_division=0))
            prec = float(precision_score(y_test, y_pred, average='weighted', zero_division=0))
            rec = float(recall_score(y_test, y_pred, average='weighted', zero_division=0))

            self.metrics = {
                'Accuracy': acc,
                'F1-score (weighted)': f1,
                'Precision (weighted)': prec,
                'Recall (weighted)': rec
            }
        else:
            r2 = float(r2_score(y_test, y_pred))
            rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
            mae = float(mean_absolute_error(y_test, y_pred))

            self.metrics = {
                'R²': r2,
                'RMSE': rmse,
                'MAE': mae
            }

        print(f"[Agente 2] Métricas en conjunto de test: {self.metrics}")
        return self.best_model_name, self.metrics, self.models_comparison


class CommunicatorAgent:
    def __init__(self, vector_db):
        self.vector_db = vector_db

    def generate_report(self, best_model_name, metrics, models_comparison, is_classification, query="partidos históricos"):
        print("[Agente 3] Generando reporte y buscando en base de datos vectorial...")

        search_results = self.vector_db.search(query, k=5)
        retrieved_text = ""
        for i, res in enumerate(search_results):
            retrieved_text += f"- {res['document']} (Relevancia: {res['score']:.2f})\n"

        problem_type = "Clasificación" if is_classification else "Regresión"
        metrics_text = "\n".join([f"   - {k}: {v:.4f}" for k, v in metrics.items()])
        comparison_text = "\n".join([f"   - {k}: {v['cv_mean_score']:.4f}" for k, v in models_comparison.items()])

        prompt = (
            f"El sistema multiagente ha completado el análisis predictivo.\n"
            f"Tipo de problema: {problem_type}\n"
            f"Mejor modelo seleccionado: {best_model_name}\n"
            f"Métricas obtenidas en test:\n{metrics_text}\n"
            f"Resultados de validación cruzada por modelo:\n{comparison_text}\n\n"
            f"Contexto de partidos del mundial recuperado de la base vectorial RAG:\n{retrieved_text}\n"
        )

        report_content = ""
        try:
            print("[Agente 3] Intentando cargar pipeline local de Transformers para redacción...")
            from transformers import pipeline
            import torch
            model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
            print(f"[Agente 3] Descargando/cargando modelo '{model_id}'...")

            # Se usa GPU si está disponible en Colab
            device = 0 if torch.cuda.is_available() else -1
            gen = pipeline("text-generation", model=model_id, device=device, max_new_tokens=600, temperature=0.7, do_sample=True)

            system_prompt = (
                "Eres un Agente Comunicador experto en fútbol e inteligencia artificial. "
                "Tu objetivo es redactar un reporte ejecutivo estructurado en español basándote en los datos predictivos proporcionados."
            )
            user_prompt = f"Por favor redacta el reporte usando los siguientes datos:\n{prompt}"

            full_prompt = f"<|system|>\n{system_prompt}</s>\n<|user|>\n{user_prompt}</s>\n<|assistant|>\n"

            print("[Agente 3] Generando reporte ejecutivo...")
            generated = gen(full_prompt)
            generated_text = generated[0]['generated_text']

            if "<|assistant|>" in generated_text:
                report_content = generated_text.split("<|assistant|>")[-1].strip()
            else:
                report_content = generated_text

        except Exception as e:
            print(f"[Agente 3] No se pudo usar Transformers local ({e}). Usando plantilla estructurada...")

            report_content = f"""# Reporte Ejecutivo: Análisis Predictivo de Partidos de los Mundiales

## 1. Resumen Ejecutivo
Este reporte detalla los resultados obtenidos por nuestro sistema multiagente para modelar y predecir resultados históricos de los mundiales.
El flujo de trabajo constó del **Agente 1 (Normalizador)**, que preparó los datos y determinó el tipo de problema, el **Agente 2 (Entrenador)**, que seleccionó el mejor modelo candidato, y el **Agente 3 (Comunicador)**, que integra contexto semántico usando una base de datos vectorial y embeddings.

---

## 2. Análisis del Rendimiento del Modelo
El problema fue identificado automáticamente como de **{problem_type.upper()}**.

### Selección del Modelo
Se evaluaron 5 modelos candidatos usando validación cruzada. El mejor modelo seleccionado fue **{best_model_name}**.

### Comparativa de Modelos (Puntuación de Validación Cruzada)
{comparison_text}

### Rendimiento en el Conjunto de Test (Datos no vistos)
{metrics_text}

---

## 3. Datos Históricos Destacados (Búsqueda Vectorial RAG)
Utilizando embeddings y similitud de coseno, el Agente 3 recuperó los siguientes partidos históricamente relevantes de la base de datos vectorial para dar contexto:

{retrieved_text}

---

## 4. Conclusiones y Próximos Pasos
1. **Idoneidad del Modelo**: El modelo **{best_model_name}** muestra ser el más consistente para generalizar y predecir los resultados de los partidos sobre este conjunto de datos.
2. **Importancia del Contexto Histórico**: La integración con la base de datos vectorial permite añadir información valiosa como la sede, el estadio y las notas históricas para explicar sorpresas del modelo.
3. **Próximos Pasos**: Integrar rankings FIFA actualizados de las selecciones antes de los mundiales para refinar la precisión.
"""

        with open('reporte_mundiales.md', 'w', encoding='utf-8') as f:
            f.write(report_content)

        return report_content

## Paso 5: Orquestador Principal y Ejecución
Cargamos el dataset, indexamos los partidos en la base vectorial, ejecutamos el pipeline y mostramos el reporte final en Markdown.

In [5]:
import pandas as pd

csv_file = 'wc_all_matches.csv'
if not os.path.exists(csv_file):
    print(f"Error: Por favor, sube primero '{csv_file}'.")
else:
    # 1. Cargar datos
    df = pd.read_csv(csv_file)
    print(f"Dataset cargado correctamente. Registros: {len(df)}")

    # 2. Indexar partidos para RAG
    print("\n=== [Fase RAG] Creando base vectorial ===")
    vector_db = VectorDB(db_path='vector_db_colab.json')

    match_texts = []
    metadatas = []
    for idx, row in df.iterrows():
        notes_str = f" Detalle: {row['notes']}." if pd.notna(row['notes']) and str(row['notes']).strip() != "" else ""
        text = (
            f"En el año {row['year']}, durante la fase {row['stage']}, el equipo de {row['team1']} "
            f"se enfrentó a {row['team2']} resultando en un marcador de {row['score1']} a {row['score2']}. "
            f"El encuentro se jugó en el estadio {row['venue']}, en la ciudad de {row['city']}, {row['country']}.{notes_str}"
        )
        match_texts.append(text)
        metadatas.append({
            'year': int(row['year']),
            'stage': str(row['stage']),
            'team1': str(row['team1']),
            'team2': str(row['team2']),
            'score1': int(row['score1']),
            'score2': int(row['score2']),
            'venue': str(row['venue']),
            'city': str(row['city']),
            'country': str(row['country'])
        })

    vector_db.add_texts(match_texts, metadatas=metadatas)
    vector_db.save()

    # 3. Preparar target y features predictivos
    df_ml = df.copy()

    def get_outcome(row):
        try:
            s1 = float(row['score1'])
            s2 = float(row['score2'])
            if s1 > s2:
                return 1
            elif s1 < s2:
                return 2
            else:
                return 0
        except:
            return 0

    df_ml['outcome'] = df_ml.apply(get_outcome, axis=1)

    # Quitar columnas de leakage
    features = ['year', 'stage', 'team1', 'team2', 'venue', 'city', 'country', 'outcome']
    df_ml = df_ml[features]

    # 4. Correr agentes
    normalizer = NormalizerAgent()
    X_clean, y_clean, is_classification = normalizer.clean_and_prepare(df_ml, target_col='outcome')

    trainer = TrainerAgent()
    best_model_name, metrics, models_comparison = trainer.train_and_evaluate(X_clean, y_clean, is_classification)

    communicator = CommunicatorAgent(vector_db)
    reporte_markdown = communicator.generate_report(
        best_model_name=best_model_name,
        metrics=metrics,
        models_comparison=models_comparison,
        is_classification=is_classification,
        query="finales históricas y partidos con muchos goles y sorpresas"
    )

Dataset cargado correctamente. Registros: 184

=== [Fase RAG] Creando base vectorial ===
[VectorDB] Cargando SentenceTransformer: all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=600) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Agente 3] Generando reporte ejecutivo...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


## Paso 6: Mostrar el Reporte Ejecutivo Generado
Renderizamos el archivo Markdown final directamente en el notebook.

In [7]:
from IPython.display import Markdown, display
display(Markdown(reporte_markdown))

En el siguiente documento, se presentan los resultados de los análisis predictivos obtenidos por el sistema de machine learning MultiAgente para la clasificación de problemas de clasificación, realizado sobre un conjunto de datos de práctica de clasificación del sistema multiagente. 

Por favor, a continuación, proporcione la información sobre la estructura de los datos y las métricas obtenidas para el análisis predictivo. Además, agrega las métricas obtenidas en el test de validación cruzada por modelo (validación cruzada por modelo) y proporcione el contexto de los partidos del mundial recuperado de la base vectorial RAG en el año 1970, en el que se enfrentaron el equipo de Brazil y el Perú. 

Si desea proporcionar más información adicional sobre los resultados de validación cruzada por modelo, puede hacerlo al final del documento.